# Raspberry Pi 5 + NoIR Camera Drowsiness Detection — MRL Dataset Training
**Dataset:** [Kaggle MRL Eye Dataset (`prasadvpatil/mrl-dataset`)](https://www.kaggle.com/datasets/prasadvpatil/mrl-dataset)

### Workflow:
1. **Runtime -> Change runtime type -> T4 GPU**.
2. **Runtime -> Run all**.
3. At the end of the notebook, `mrl_eye_model.onnx` (and `mrl_eye_model.pt`) will automatically download to your computer.
4. Place `mrl_eye_model.onnx` in the same folder as `app.py` on your Raspberry Pi 5 (or PC) and run `python app.py`.

In [ ]:
!pip install -q kagglehub onnx opencv-python-headless scikit-learn matplotlib tqdm

In [ ]:
import os
import random
from pathlib import Path

import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Download the MRL Eye Dataset from Kaggle
`kagglehub` downloads `prasadvpatil/mrl-dataset` directly without needing to upload `kaggle.json`.

In [ ]:
dataset_root = Path(kagglehub.dataset_download("prasadvpatil/mrl-dataset"))
print(f"Dataset downloaded to: {dataset_root}")

# Discover all image files and map parent folder names:
# Class 0: Close-Eyes (Closed)
# Class 1: Open-Eyes  (Open)
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}

train_samples = []
test_samples = []
all_samples = []

for img_path in dataset_root.rglob("*"):
    if img_path.suffix.lower() not in IMAGE_EXTS:
        continue
    parts_lower = [p.lower() for p in img_path.parts]
    parent_lower = img_path.parent.name.lower()

    if "close" in parent_lower:
        label = 0  # Close-Eyes
    elif "open" in parent_lower:
        label = 1  # Open-Eyes
    else:
        continue

    all_samples.append((str(img_path), label))
    if any("test" in p for p in parts_lower):
        test_samples.append((str(img_path), label))
    elif any("train" in p for p in parts_lower):
        train_samples.append((str(img_path), label))

# If dataset has explicit train/test folders, split train into train + val
if len(train_samples) > 0 and len(test_samples) > 0:
    labels = [s[1] for s in train_samples]
    train_list, val_list = train_test_split(
        train_samples, test_size=0.15, random_state=SEED, stratify=labels
    )
    test_list = test_samples
else:
    labels = [s[1] for s in all_samples]
    train_val, test_list = train_test_split(
        all_samples, test_size=0.15, random_state=SEED, stratify=labels
    )
    tv_labels = [s[1] for s in train_val]
    train_list, val_list = train_test_split(
        train_val, test_size=0.15, random_state=SEED, stratify=tv_labels
    )

print(f"Total images found : {len(all_samples)}")
print(f"Train split        : {len(train_list)} (Closed={sum(x[1]==0 for x in train_list)}, Open={sum(x[1]==1 for x in train_list)})")
print(f"Val split          : {len(val_list)} (Closed={sum(x[1]==0 for x in val_list)}, Open={sum(x[1]==1 for x in val_list)})")
print(f"Test split         : {len(test_list)} (Closed={sum(x[1]==0 for x in test_list)}, Open={sum(x[1]==1 for x in test_list)})")

## 2. NoIR-Matched Dataset & Preprocessing Pipeline
Raspberry Pi NoIR cameras have no IR-cut filter (producing pinkish RGB images in daylight and monochrome NIR images under IR LEDs at night). By converting all crops to **1-channel Grayscale + CLAHE (`clipLimit=2.0, tileGridSize=(4, 4)`)**, the live camera frames on the Pi 5 match the MRL NIR dataset distribution under both day and night lighting.

In [ ]:
IMG_SIZE = 64

class MRLEyeDataset(Dataset):
    def __init__(self, samples, is_train: bool = False):
        self.samples = samples
        self.is_train = is_train
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))

    def __len__(self):
        return len(self.samples)

    def _augment(self, img: np.ndarray) -> np.ndarray:
        # 1. Random horizontal flip
        if random.random() < 0.5:
            img = cv2.flip(img, 1)

        # 2. Random rotation (-15 to +15 deg) & slight scale (0.90 to 1.10)
        if random.random() < 0.7:
            angle = random.uniform(-15.0, 15.0)
            scale = random.uniform(0.90, 1.10)
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
            img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

        # 3. Random IR brightness / contrast shift (simulates NoIR LED distance changes)
        if random.random() < 0.6:
            alpha = random.uniform(0.75, 1.25)
            beta = random.uniform(-20, 20)
            img = np.clip(alpha * img.astype(np.float32) + beta, 0, 255).astype(np.uint8)

        # 4. Occasional slight Gaussian blur (motion / focus tolerance)
        if random.random() < 0.2:
            img = cv2.GaussianBlur(img, (3, 3), 0)

        return img

    def __getitem__(self, idx: int):
        img_path, label = self.samples[idx]
        gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if gray is None:
            gray = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

        if self.is_train:
            gray = self._augment(gray)

        # Exact same preprocessing as app.py on Raspberry Pi 5:
        eq = self.clahe.apply(gray)
        resized = cv2.resize(eq, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        norm = (resized.astype(np.float32) / 255.0 - 0.5) / 0.5

        tensor = torch.from_numpy(norm).unsqueeze(0)  # (1, 64, 64)
        return tensor, torch.tensor(label, dtype=torch.long)

BATCH_SIZE = 64
train_ds = MRLEyeDataset(train_list, is_train=True)
val_ds = MRLEyeDataset(val_list, is_train=False)
test_ds = MRLEyeDataset(test_list, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Preview sample preprocessed eye crops
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    x, y = train_ds[i]
    ax.imshow(x.squeeze(0).numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title("Closed (0)" if y.item() == 0 else "Open (1)")
    ax.axis("off")
plt.suptitle("CLAHE Preprocessed MRL Eye Samples (64x64 Grayscale)")
plt.tight_layout()
plt.show()

## 3. Define `MRLEyeCNN` Architecture & Train on GPU
This architecture matches `MRLEyeCNN` in `app.py` and runs in **< 1.5 ms** per eye on the Raspberry Pi 5 CPU.

In [ ]:
class MRLEyeCNN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 64x64 -> 32x32
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),
            # Block 2: 32x32 -> 16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.20),
            # Block 3: 16x16 -> 8x8
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            # Block 4: 8x8 -> 1x1
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

model = MRLEyeCNN(num_classes=2).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

In [ ]:
EPOCHS = 20
criterion = nn.CrossEntropyLoss(label_smoothing=0.03)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0
best_ckpt_path = "mrl_eye_model.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [Train]", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        t_loss += loss.item() * x.size(0)
        t_correct += (logits.argmax(dim=1) == y).sum().item()
        t_total += x.size(0)

    scheduler.step()

    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            v_loss += loss.item() * x.size(0)
            v_correct += (logits.argmax(dim=1) == y).sum().item()
            v_total += x.size(0)

    train_loss = t_loss / t_total
    train_acc = t_correct / t_total
    val_loss = v_loss / v_total
    val_acc = v_correct / v_total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    saved_tag = ""
    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_ckpt_path)
        saved_tag = " [*BEST*]"

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f}, Acc: {val_acc*100:.2f}%{saved_tag}"
    )

print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}%")

## 4. Evaluate on Test Set & Plot Metrics

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="Train Loss")
ax1.plot(history["val_loss"], label="Val Loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot([a * 100 for a in history["train_acc"]], label="Train Acc (%)")
ax2.plot([a * 100 for a in history["val_acc"]], label="Val Acc (%)")
ax2.set_title("Accuracy (%)")
ax2.legend()
plt.show()

# Load best checkpoint and evaluate on test split
model.load_state_dict(torch.load(best_ckpt_path, map_location=device, weights_only=True))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for x, y in test_loader:
        logits = model(x.to(device))
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(y.numpy())

print("\n=== Test Set Classification Report ===")
print(classification_report(all_targets, all_preds, target_names=["Close-Eyes (0)", "Open-Eyes (1)"], digits=4))
print("Confusion Matrix:")
print(confusion_matrix(all_targets, all_preds))

## 5. Export to ONNX for Raspberry Pi 5 (`cv2.dnn` Ready) & Download
Exports a single self-contained `mrl_eye_model.onnx` file and verifies that OpenCV's `cv2.dnn.readNetFromONNX` can run it directly.

In [ ]:
onnx_path = "mrl_eye_model.onnx"
model_cpu = model.to("cpu").eval()
dummy_input = torch.randn(1, 1, IMG_SIZE, IMG_SIZE, dtype=torch.float32)

# Use legacy TorchScript exporter (dynamo=False) so all weights are embedded in a single .onnx file
# and 100% compatible with OpenCV cv2.dnn.readNetFromONNX on Raspberry Pi 5.
try:
    torch.onnx.export(
        model_cpu,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=13,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["logits"],
        dynamo=False,
    )
except TypeError:
    torch.onnx.export(
        model_cpu,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=13,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["logits"],
    )

# Verify with OpenCV DNN (exact backend used by app.py on Raspberry Pi 5)
net = cv2.dnn.readNetFromONNX(onnx_path)
net.setInput(dummy_input.numpy())
cv_out = net.forward()
with torch.no_grad():
    pt_out = model_cpu(dummy_input).numpy()

max_diff = float(np.max(np.abs(cv_out - pt_out)))
print(f"ONNX exported successfully: {onnx_path} ({os.path.getsize(onnx_path) / 1024:.1f} KB)")
print(f"OpenCV DNN verification max diff vs PyTorch: {max_diff:.6e}")

# Auto-download in Google Colab
try:
    from google.colab import files
    files.download(onnx_path)
    files.download(best_ckpt_path)
except ImportError:
    print("Not running inside Google Colab UI; files saved locally.")